In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv(r'C:\Users\Pc\Documents\GitHub\autoinsightcs70\model_training\clean#1 (1).csv')
print(data.head())

   Unnamed: 0 Vehicle Type    Make                Model    Year      Price  \
0           0          Car  Toyota                 Axio  2008.0  7150000.0   
1           1          Car  Suzuki              Wagon R  2016.0  5690000.0   
2           2          Car  Toyota  VITZ SAFETY EDITION  2016.0        NaN   
3           3          Car  Suzuki           Alto Japan  2023.0  6590000.0   
4           4          Car   Honda             EK3 ViRs  2000.0        NaN   

   Milleage      District published date  \
0  178000.0       Colombo       2/1/2026   
1   79000.0        Ragama       2/7/2026   
2   85000.0        Chilaw       2/7/2026   
3    9900.0  Kiribathgoda       2/7/2026   
4     264.0         Kandy       2/7/2026   

                                         Vehicle URL  
0  https://riyasewana.com/buy/toyota-axio-sale-co...  
1  https://riyasewana.com/buy/suzuki-wagon-r-sale...  
2  https://riyasewana.com/buy/toyota-vitz-safety-...  
3  https://riyasewana.com/buy/suzuki-alto-japa

In [2]:
print(data.isnull().sum())

Unnamed: 0           0
Vehicle Type         0
Make                 0
Model               25
Year                42
Price             7096
Milleage          3986
District             1
published date       0
Vehicle URL          0
dtype: int64


In [3]:
print(data['Milleage'].describe())

count    2.969800e+04
mean     5.419231e+05
std      1.818768e+07
min      1.000000e+00
25%      6.600000e+04
50%      1.215000e+05
75%      1.740000e+05
max      1.234568e+09
Name: Milleage, dtype: float64


In [4]:

data = data[data['Price'] < 30000000].copy()
data = data[data['Milleage'] < 1000000].copy()

print('Rows after filtering: ' , data.shape[0])

Rows after filtering:  23430


In [5]:
def get_condition(year, milleage):
    if year < 2022:
        return 'Used'
    else:
        if pd.isnull(milleage):
            return 'Unknown'
        elif milleage < 5000:
            return 'Brand New'
        elif milleage <= 50000:
            return 'Recondition'
        else: 
            return 'Used'

# Create a new column 'Condition' based on the 'Year' and 'Milleage' columns    
data['Condition'] = data.apply(lambda row: get_condition(row['Year'], row['Milleage']), axis=1)   

print (data['Condition'].value_counts())
print(data[['Year', 'Milleage', 'Condition']].head())

Condition
Used           20926
Recondition     1398
Brand New       1106
Name: count, dtype: int64
     Year  Milleage    Condition
0  2008.0  178000.0         Used
1  2016.0   79000.0         Used
3  2023.0    9900.0  Recondition
5  2011.0  156100.0         Used
6  2013.0   76000.0         Used


In [6]:
data.to_csv('dataset_with_condition.csv', index=False)
print('Saved! Columns: ', data.columns.tolist())

Saved! Columns:  ['Unnamed: 0', 'Vehicle Type', 'Make', 'Model', 'Year', 'Price', 'Milleage', 'District', 'published date', 'Vehicle URL', 'Condition']


In [7]:
data1 = pd.read_csv(r'C:\Users\Pc\Documents\GitHub\autoinsightcs70\model_training\dataset_with_condition.csv')
print(data1.isnull().sum())

Unnamed: 0         0
Vehicle Type       0
Make               0
Model             15
Year              23
Price              0
Milleage           0
District           0
published date     0
Vehicle URL        0
Condition          0
dtype: int64


In [8]:

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import joblib
import warnings
warnings.filterwarnings('ignore')


# 1. DATA LOADING & CLEANING


def load_and_clean(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Drop unnamed index column if present
    df = df.drop(columns=[c for c in df.columns if 'Unnamed' in c], errors='ignore')

    # Drop rows with missing Model or Year (can't meaningfully use them)
    df = df.dropna(subset=['Model', 'Year', 'Price', 'Milleage'])

    # Type casting
    df['Year'] = df['Year'].astype(int)
    df['published date'] = pd.to_datetime(df['published date'])

    # Remove extreme outliers (already done in notebook, kept here for safety)
    df = df[df['Price'] < 30000000]
    df = df[df['Milleage'] < 1000000]

    # Sort for lag feature correctness — CRITICAL
    df = df.sort_values(['Make', 'Model', 'published date']).reset_index(drop=True)

    return df


# 2. FEATURE ENGINEERING


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['post_year'] = df['published date'].dt.year
    df['Car_Age'] = df['post_year'] - df['Year']
    df['month'] = df['published date'].dt.month
    df['week'] = df['published date'].dt.isocalendar().week.astype(int)
    df['day_of_year'] = df['published date'].dt.dayofyear

    # Lag features (within Make+Model group) — lag_1 = last known price
    df['lag_1'] = df.groupby(['Make', 'Model'])['Price'].shift(1)
    df['lag_2'] = df.groupby(['Make', 'Model'])['Price'].shift(2)

    # Rolling mean of last 3 prices for the same Make+Model
    df['rolling_mean_3'] = (
        df.groupby(['Make', 'Model'])['Price']
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

    # Drop rows where lag features are NaN (first entry per Make+Model)
    df = df.dropna(subset=['lag_1', 'lag_2', 'rolling_mean_3'])

    return df


# 3. MODEL TRAINING


NUMERIC_FEATURES = [
    'Car_Age',
    'month',
    'week',
    'day_of_year',
    'lag_1',
    'lag_2',
    'rolling_mean_3',
]
CATEGORICAL_FEATURES = ['Make', 'Model']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def train_model(df: pd.DataFrame):
    df = df.copy()

    # Encode categoricals for LightGBM native handling
    for col in CATEGORICAL_FEATURES:
        df[col] = df[col].astype('category')

    # Temporal train/test split — last 30 days as test (mimics production)
    split_date = df['published date'].max() - pd.Timedelta(days=30)
    train = df[df['published date'] <= split_date]
    test  = df[df['published date'] >  split_date]

    X_train, y_train = train[ALL_FEATURES], train['Price']
    X_test,  y_test  = test[ALL_FEATURES],  test['Price']

    print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")

    model = lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        verbose=-1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        categorical_feature=CATEGORICAL_FEATURES,
        callbacks=[
            lgb.early_stopping(stopping_rounds=150, verbose=False),
            lgb.log_evaluation(period=200),
        ]
    )

    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    mape  = mean_absolute_percentage_error(y_test, preds) * 100

    print(f"\n{'='*45}")
    print(f"  Test MAE  : LKR {mae:,.0f}")
    print(f"  Test MAPE : {mape:.2f}%")
    print(f"{'='*45}\n")

    return model, df



# 4. KEY FIX: USER-DATE BASED PREDICTION


def predict_from_user_date(
    model,
    df: pd.DataFrame,
    make: str,
    model_name: str,
    user_query_date: str,
    manufacture_year: int,
    horizon_days: int = 7,   # 7 = next week, 28 = next month
    verbose: bool = True,     # Control output verbosity
) -> dict:
  

    query_date = pd.Timestamp(user_query_date)

    # Step 1: Get the most recent known price for this Make+Model
    subset = df[
        (df['Make'].str.lower() == make.lower()) &
        (df['Model'].str.lower() == model_name.lower()) &
        (df['published date'] <= query_date)
    ].sort_values('published date')

    if subset.empty:
        # Fallback: use any listing for this Make+Model regardless of date
        subset = df[
            (df['Make'].str.lower() == make.lower()) &
            (df['Model'].str.lower() == model_name.lower())
        ].sort_values('published date')

    if subset.empty:
        raise ValueError(f"No data found for {make} {model_name}. "
                         f"Check spelling or add data for this vehicle.")

    last_known_price    = subset.iloc[-1]['Price']
    second_last_price   = subset.iloc[-2]['Price'] if len(subset) >= 2 else last_known_price
    rolling_mean_price  = subset['Price'].tail(3).mean()

    if verbose:
        print(f"\n  Vehicle      : {make} {model_name} ({manufacture_year})")
        print(f"  Query Date   : {query_date.date()}")
        print(f"  Last Known Price (as of query date): LKR {last_known_price:,.0f}")
        print(f"  Forecasting  : {horizon_days} days ahead\n")

    # Step 2: Build a synthetic "current" row using user-supplied vehicle info
    car_age = query_date.year - manufacture_year

    current_row = {
        'Make'             : make,
        'Model'            : model_name,
        'Car_Age'          : car_age,
        'lag_1'            : last_known_price,
        'lag_2'            : second_last_price,
        'rolling_mean_3'   : rolling_mean_price,
    }

    # Step 3: Iterative weekly rollout from user_query_date
    predictions = {}
    future_row  = current_row.copy()
    future_date = query_date

    steps = max(1, horizon_days // 7)  # weekly steps

    for _ in range(1, steps + 1):
        # Calculate the start and end of this week period
        week_start = future_date + pd.Timedelta(days=7)
        week_end = week_start + pd.Timedelta(days=6)
        
        future_date = week_start

        future_row['month']       = future_date.month
        future_row['week']        = future_date.isocalendar()[1]
        future_row['day_of_year'] = future_date.timetuple().tm_yday

        # Build prediction dataframe with proper dtypes
        pred_df = pd.DataFrame([future_row])
        for col in CATEGORICAL_FEATURES:
            # Use the full category set from training data
            known_cats = df[col].astype('category').cat.categories
            pred_df[col] = pd.Categorical(pred_df[col], categories=known_cats)

        price_pred = model.predict(pred_df[ALL_FEATURES])[0]

        # Store with date range to show price is constant for the entire week
        date_range = f"{week_start.strftime('%Y-%m-%d')} to {week_end.strftime('%Y-%m-%d')}"
        predictions[date_range] = round(price_pred, 2)

        # Update lag for next iteration (the key to chained forecasting)
        future_row['lag_2']           = future_row['lag_1']
        future_row['lag_1']           = price_pred
        future_row['rolling_mean_3']  = np.mean([
            future_row['lag_1'],
            future_row['lag_2'],
            rolling_mean_price
        ])

    return {
        'make'             : make,
        'model'            : model_name,
        'query_date'       : query_date.strftime('%Y-%m-%d'),
        'current_price'    : last_known_price,
        'predictions'      : predictions,
    }



# 5. CONVENIENCE WRAPPERS


def predict_next_week(model, df, make, model_name, user_query_date,
                      manufacture_year):
    return predict_from_user_date(
        model, df, make, model_name, user_query_date,
        manufacture_year, horizon_days=7
    )


def predict_next_month(model, df, make, model_name, user_query_date,
                       manufacture_year):
    return predict_from_user_date(
        model, df, make, model_name, user_query_date,
        manufacture_year, horizon_days=28
    )



# 7. MAIN — TRAIN + GENERATE CSV + DEMO


def generate_vehicle_statistics_csv(
    model,
    df: pd.DataFrame,
    raw_df: pd.DataFrame,
    output_path: str = 'vehicle_statistics_with_predictions.csv'
):
    print("\n" + "="*60)
    print("  GENERATING VEHICLE STATISTICS CSV")
    print("="*60)
    
    # Group by Make, Model, Year and calculate statistics
    vehicle_groups = raw_df.groupby(['Make', 'Model', 'Year']).agg({
        'Price': 'mean',
        'Milleage': 'mean'
    }).reset_index()
    
    vehicle_groups.columns = ['Make', 'Model', 'Year', 'Average_Price', 'Average_Mileage']
    
    # Calculate predicted next week price for each vehicle
    print(f"\n  Calculating predictions for {len(vehicle_groups)} vehicles...")
    predicted_prices = []
    
    for idx, row in vehicle_groups.iterrows():
        try:
            # Use the most recent date in the dataset as reference
            latest_date = df['published date'].max()
            
            result = predict_from_user_date(
                model, df,
                make=row['Make'],
                model_name=row['Model'],
                user_query_date=latest_date,
                manufacture_year=int(row['Year']),
                horizon_days=7,
                verbose=False  # Suppress detailed output
            )
            
            # Get the first predicted price (next week)
            first_prediction = list(result['predictions'].values())[0]
            predicted_prices.append(first_prediction)
            
            if (idx + 1) % 50 == 0:
                print(f"    Processed {idx + 1}/{len(vehicle_groups)} vehicles...")
                
        except Exception as e:
            # If prediction fails, use average price
            predicted_prices.append(row['Average_Price'])
    
    vehicle_groups['Predicted_Next_Week_Price'] = predicted_prices
    
    # Round values for cleaner output
    vehicle_groups['Average_Price'] = vehicle_groups['Average_Price'].round(0).astype(int)
    vehicle_groups['Average_Mileage'] = vehicle_groups['Average_Mileage'].round(0).astype(int)
    vehicle_groups['Predicted_Next_Week_Price'] = vehicle_groups['Predicted_Next_Week_Price'].round(0).astype(int)
    
    # Sort by Make, Model, Year
    vehicle_groups = vehicle_groups.sort_values(['Make', 'Model', 'Year']).reset_index(drop=True)
    
    # Save to CSV
    vehicle_groups.to_csv(output_path, index=False)
    
    print(f"\n  ✓ CSV file generated successfully!")
    print(f"  ✓ Location: {output_path}")
    print(f"  ✓ Total vehicles: {len(vehicle_groups)}")
    print(f"\n  Sample output (first 5 rows):")
    print(vehicle_groups.head().to_string(index=False))
    print("\n" + "="*60)
    
    return vehicle_groups



# 6. MAIN — TRAIN + DEMO

if __name__ == '__main__':
    CSV_PATH = r'C:\Users\Pc\Documents\GitHub\autoinsightcs70\model_training\dataset_with_condition.csv'
    
    print("Loading data...")
    raw_df = load_and_clean(CSV_PATH)
    
    print("Engineering features...")
    eng_df = engineer_features(raw_df)
    
    print("Training model...")
    trained_model, eng_df = train_model(eng_df)
    
    # Save model for API/backend use
    joblib.dump(trained_model, 'lgbm_vehicle_price_model.pkl')
    print("Model saved to lgbm_vehicle_price_model.pkl\n")
    
    # ── GENERATE CSV WITH VEHICLE STATISTICS
    vehicle_stats = generate_vehicle_statistics_csv(
        trained_model, eng_df, raw_df,
        output_path='vehicle_statistics_with_predictions.csv'
    )
    
    # ── DEMO: User inputs only Make, Model, and Year
    #          System automatically uses the date and finds prices
    print("\n" + "="*60)
    print("  DEMO: VEHICLE PRICE PREDICTION")
    print("="*60)
    
    USER_QUERY_DATE  = '2026-03-01'   # ← This is the date the USER opens the app
    MAKE             = 'Suzuki'
    MODEL_NAME       = 'Wagon R'
    MANUFACTURE_YEAR = 2017
    
    print("\n" + "="*50)
    print("  NEXT WEEK FORECAST")
    print("="*50)
    
    week_result = predict_next_week(
        trained_model, eng_df,
        MAKE, MODEL_NAME, USER_QUERY_DATE,
        MANUFACTURE_YEAR
    )
    
    print(f"\n  Current Price (Today): LKR {week_result['current_price']:>15,.0f}")
    print(f"  (This price is valid for the next 7 days)\n")
    print(f"  Predicted Prices (Price remains constant for entire week):")
    
    for date_range, price in week_result['predictions'].items():
        print(f"  {date_range}  →  LKR {price:>12,.0f}")
    
    print("\n" + "="*50)
    print("  NEXT MONTH FORECAST (Weekly Steps)")
    print("="*50)
    
    month_result = predict_next_month(
        trained_model, eng_df,
        MAKE, MODEL_NAME, USER_QUERY_DATE,
        MANUFACTURE_YEAR
    )
    
    print(f"\n  Current Price (Today): LKR {month_result['current_price']:>15,.0f}")
    print(f"  (This price is valid for the next 7 days)\n")
    print(f"  Predicted Prices (Price remains constant for entire week):")
    
    for date_range, price in month_result['predictions'].items():
        print(f"  {date_range}  →  LKR {price:>12,.0f}")
    
    print("\n" + "="*60)

    

Loading data...
Engineering features...
Training model...
Training samples: 5795 | Test samples: 10575
[200]	valid_0's l2: 3.20673e+12
[400]	valid_0's l2: 3.19801e+12

  Test MAE  : LKR 916,739
  Test MAPE : 24.21%

Model saved to lgbm_vehicle_price_model.pkl


  GENERATING VEHICLE STATISTICS CSV

  Calculating predictions for 9904 vehicles...
    Processed 100/9904 vehicles...
    Processed 350/9904 vehicles...
    Processed 500/9904 vehicles...
    Processed 650/9904 vehicles...
    Processed 850/9904 vehicles...
    Processed 900/9904 vehicles...
    Processed 1000/9904 vehicles...
    Processed 1050/9904 vehicles...
    Processed 1100/9904 vehicles...
    Processed 1200/9904 vehicles...
    Processed 1250/9904 vehicles...
    Processed 1300/9904 vehicles...
    Processed 1350/9904 vehicles...
    Processed 1400/9904 vehicles...
    Processed 1450/9904 vehicles...
    Processed 1500/9904 vehicles...
    Processed 1600/9904 vehicles...
    Processed 1650/9904 vehicles...
    Processe